# gd-levelgen — Colab worker (training the Geometry Dash level model)

Run this notebook and leave the tab open; training jobs are queued from the local machine (`python colab/jobs.py add ...`).

1. Runtime → Change runtime type → **H100** (or A100 / RTX 6000) GPU.
2. 🔑 Secrets (Notebook access on): `KAGGLE_API_TOKEN` = contents of `~/.kaggle/access_token`.
3. Run the single code cell (or Runtime → Run all). Keep **one** tab with this notebook: two workers A would take the same jobs. No Google Drive: everything persistent is a private Kaggle dataset (`gdgen-*`).

First start of a session takes ~5–10 min: the pretokenized levels (outputs of the Kaggle `gd-levelgen-tok-*` kernels, several GB) are downloaded to the runtime disk. Then the worker polls `kirillkholkin/gdgen-jobs` every 2 min, runs jobs and publishes status, log tails and small results; run dirs (checkpoints) are saved to `gdgen-runs-<name>` while a job runs, so after a disconnect just Run all again — training resumes from the last checkpoint. After 15 min without jobs the runtime is released (no compute units burn while idle).

In [ ]:
from google.colab import userdata
import os, subprocess
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
tok = os.path.expanduser('~/.kaggle/access_token')
open(tok, 'w').write(os.environ['KAGGLE_API_TOKEN']); os.chmod(tok, 0o600)
subprocess.run('pip install -q -U "kaggle>=2.0"', shell=True, check=True)
subprocess.run('rm -rf /content/code && kaggle datasets download kirillkholkin/gd-levelgen-code -p /content/code --unzip -q',
               shell=True, check=True)
print('code version', open('/content/code/CODE_VERSION').read().strip())
subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; free -g | head -2; df -h /content | tail -1', shell=True)

# one cell on purpose: the worker must never start without a fresh copy of the code
%run /content/code/colab/worker.py --worker A